# Chapter 6: Drift Monitoring

## The Smoke Detector for When the World Changes

Your model was trained on customers who looked a certain way — maybe average tenure was 24 months, most were on annual contracts, and typical monthly charges were $60.

Six months later, you're scoring a new batch. But now average tenure is 6 months, everyone is month-to-month, and charges average $110. **The world changed.** The model learned patterns from a different reality — its predictions may no longer be reliable.

How do you catch this BEFORE bad predictions reach the client? That's what drift monitoring does.

**PSI (Population Stability Index)** is our smoke detector. It compares the distribution of each feature in today's data against the training data and says: "Same world, or different world?"

In [ ]:
import numpy as np
import pandas as pd
from churn_pipeline.steps.monitoring import compute_psi, check_drift, DriftReport

## How PSI Works (Step by Step)

1. **Divide the training data into buckets.** For tenure, maybe: 0-12 months, 12-24, 24-36, etc.
2. **Count what % of training customers fall in each bucket.** Maybe 30% in 0-12, 40% in 12-36, 30% in 36+.
3. **Count what % of NEW customers fall in the same buckets.** Maybe now 60% are in 0-12!
4. **Compute divergence per bucket:** (new% - old%) × ln(new% / old%)
5. **Sum across all buckets** → that's PSI.

### Interpretation

| PSI Value | Meaning | Action |
|-----------|---------|--------|
| < 0.1 | Stable — world looks the same | Safe, continue scoring |
| 0.1 - 0.2 | Moderate shift — something changed | Watch closely, investigate |
| > 0.2 | Significant change — different world | Alert! Consider retraining |

## Demo 1: No Drift (Same Distribution)

When training data and current data come from the same distribution, PSI should be ~0.

In [ ]:
np.random.seed(42)

# Training data: tenure averaging 24 months
training_tenure = np.random.normal(24, 12, size=1000)

# Current data: SAME distribution (no drift)
current_tenure_same = np.random.normal(24, 12, size=300)

psi_no_drift = compute_psi(training_tenure, current_tenure_same)
print(f"PSI (no drift): {psi_no_drift:.6f}")
print(f"Interpretation: {'Stable ✓' if psi_no_drift < 0.1 else 'DRIFT!'}")

## Demo 2: Moderate Drift (Small Shift)

In [ ]:
# Current data: tenure shifted down by 5 months (new customers joining)
current_tenure_moderate = np.random.normal(19, 12, size=300)

psi_moderate = compute_psi(training_tenure, current_tenure_moderate)
print(f"PSI (moderate drift): {psi_moderate:.4f}")
print(f"Interpretation: {'Moderate shift ⚠️' if 0.1 <= psi_moderate < 0.2 else 'Stable' if psi_moderate < 0.1 else 'Significant! 🚨'}")

## Demo 3: Significant Drift (World Changed)

In [ ]:
# Current data: tenure dramatically lower (mass new customer acquisition)
current_tenure_drift = np.random.normal(8, 5, size=300)

psi_drift = compute_psi(training_tenure, current_tenure_drift)
print(f"PSI (significant drift): {psi_drift:.4f}")
print(f"Interpretation: SIGNIFICANT DRIFT 🚨 — model may be unreliable!")
print(f"\nWhat happened: training saw avg tenure ~24 months,")
print(f"but current batch has avg tenure ~8 months.")
print(f"The model learned patterns about long-tenure customers")
print(f"and is now being asked about short-tenure ones.")

## Demo 4: The Self-Comparison Sanity Check

A critical property: comparing any distribution against itself must produce PSI = 0. If something hasn't changed, the smoke detector must report zero smoke. This is how we know our implementation is correct.

In [ ]:
# PSI of data against itself = always 0
data = np.random.normal(50, 15, size=500)
psi_self = compute_psi(data, data)
print(f"PSI (self-comparison): {psi_self:.15f}")
print(f"Equal to zero within floating-point precision: {abs(psi_self) < 1e-10} ✓")

## Demo 5: Multi-Feature Drift Report

In practice, we check ALL features at once. The `check_drift` function computes PSI for every tracked feature and raises an alert if ANY feature exceeds the threshold.

In [ ]:
# Training distributions (what the model was trained on)
training_stats = {
    "tenure_months": np.random.normal(24, 12, size=1000),
    "monthly_charges": np.random.normal(60, 20, size=1000),
    "total_charges": np.random.normal(1500, 800, size=1000),
    "support_tickets": np.random.poisson(2, size=1000).astype(float),
}

# Scenario: tenure drifted significantly, charges shifted slightly, tickets stable
current_data = pd.DataFrame({
    "tenure_months": np.random.normal(10, 6, size=300),       # BIG drift
    "monthly_charges": np.random.normal(65, 22, size=300),    # Slight shift
    "total_charges": np.random.normal(1400, 900, size=300),   # Stable
    "support_tickets": np.random.poisson(2, size=300).astype(float),  # Stable
})

report = check_drift(training_stats, current_data, threshold=0.2)

print("Drift Report")
print("=" * 50)
print(f"Features checked: {report.features_checked}")
print(f"Alert triggered:  {report.alert_triggered}")
print(f"\nPer-feature PSI scores:")
for feature, psi in sorted(report.psi_scores.items(), key=lambda x: -x[1]):
    status = "🚨 DRIFTED" if feature in report.features_drifted else "✓ stable"
    print(f"  {feature:<20} PSI = {psi:.4f}  {status}")

if report.features_drifted:
    print(f"\n⚠️  Features requiring attention: {report.features_drifted}")
    print(f"    Action: Investigate why these distributions changed.")
    print(f"    Consider retraining if the shift is permanent.")

## When Drift Is NOT a Problem

Drift doesn't automatically mean the model is wrong. It means **you should check:**

- **Seasonal drift:** Summer customers look different from winter customers. If your model was trained on full-year data, this is expected and harmless.
- **One-time campaign:** A marketing push brought in many new customers (low tenure). This is temporary — next month's batch will normalize.
- **Real shift:** The business changed — new pricing, new product. The model needs retraining.

PSI tells you "something changed." You decide if it matters.

## Key Takeaways

1. **Drift = the world changed** — data no longer looks like what the model was trained on
2. **PSI quantifies the change** — 0 = identical, >0.2 = significant shift
3. **Self-comparison = 0** — the sanity check that proves the math works
4. **Check ALL features** — one drifted feature can make the whole model unreliable
5. **Alert, don't halt** — drift monitoring triggers notifications but doesn't stop scoring (the model might still be useful)
6. **Human judgment required** — PSI says "something changed," you decide what to do

Next: Chapter 7 covers LLM integration — using Claude to automate the boring parts of ML ops.

---

*Source code: `src/churn_pipeline/steps/monitoring.py`*  
*Tests: `tests/property/test_monitoring.py`, `tests/unit/test_monitoring.py`*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*